# Kaggle runner: train one experiment

Runs `python -m trainer.train` for one experiment file in `configs/experiment/` and packs the results.
It also runs locally (it then uses the project folder and your `.venv`).

**Kaggle notebook settings**
- **Accelerator:** GPU **T4** (the code uses one GPU; recent PyTorch builds may not support the P100).
- **Internet: On** (backbones download from Hugging Face; `pip install`).
- **Code:** either set `GIT_URL` below, or attach the project folder as a Kaggle Dataset
  (without `.venv/` and `models/`). `data/processed/` must be included: training reads it.
- **Keep results:** outputs live in `/kaggle/working` (~20 GB). Use *Save Version* so they persist,
  or download the zip written at the end.

**Resume:** sessions stop after ~12 h. Run the notebook again (same experiment): finished runs
(`metrics.json` present) are skipped. Keep `/kaggle/working/models` between sessions by attaching the
previous version's output as a Dataset and copying it back (see the resume cell).

## 1. Settings
The only cell to edit.

In [ ]:
EXPERIMENT = 'e1_baseline'    # file name in configs/experiment/ (without .yaml)
ONLY = None                   # substring filter on run ids, e.g. 'uit-vsfc__' or '__phobert__'
MAX_STEPS = None              # e.g. 20: quick check, outputs go to models/_debug/
GIT_URL = None                # e.g. 'https://github.com/<user>/DPL_project.git'; None = attached Kaggle Dataset
GIT_BRANCH = 'master'
RESUME_FROM = None            # folder with a previous session's models/ (e.g. '/kaggle/input/<prev-output>/models')
RUN_PREPROCESS = False        # regenerate data/processed on Kaggle (e.g. with VnCoreNLP for PhoBERT)
INSTALL_VNCORENLP = False     # needs Java; only used with RUN_PREPROCESS
ZIP_RESULTS = True            # results_<experiment>.zip without checkpoints (Kaggle only)

## 2. Find the code
Clones `GIT_URL` into `/kaggle/working`, or finds the project in the attached Datasets
(a folder with `trainer/train.py` and `utils/dataloader.py`). Locally: the project this notebook is in
(or `$DPL_PROJECT_ROOT`).

In [ ]:
import os, sys, json, shutil, subprocess, zipfile
from pathlib import Path

ON_KAGGLE = os.path.isdir('/kaggle/input')
WORK = Path('/kaggle/working')


def is_project(p):
    return (Path(p) / 'trainer' / 'train.py').is_file() and (Path(p) / 'utils' / 'dataloader.py').is_file()


def find_project(root, max_depth=4):
    level = [Path(root)]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            if is_project(d):
                return d
            try:
                nxt += sorted(c for c in d.iterdir() if c.is_dir())
            except OSError:
                pass
        level = nxt
    return None


if os.environ.get('DPL_PROJECT_ROOT') and is_project(os.environ['DPL_PROJECT_ROOT']):
    CODE = Path(os.environ['DPL_PROJECT_ROOT']).resolve()
elif ON_KAGGLE and GIT_URL:
    CODE = WORK / 'DPL_project'
    if CODE.exists():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', GIT_BRANCH, GIT_URL, str(CODE)], check=True)
elif ON_KAGGLE:
    CODE = find_project('/kaggle/input')
    if CODE is None:
        raise FileNotFoundError('Project not found in /kaggle/input: attach it as a Dataset or set GIT_URL.')
else:
    CODE = next((p for p in (Path.cwd(), *Path.cwd().parents[:2]) if is_project(p)), None)
    if CODE is None:
        raise FileNotFoundError('Run this notebook from inside the project.')

os.chdir(CODE)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
sys.dont_write_bytecode = True      # the attached Dataset is read-only
EXP_FILE = CODE / 'configs' / 'experiment' / f'{EXPERIMENT}.yaml'
assert EXP_FILE.is_file(), f'{EXP_FILE} not found; available: {sorted(p.stem for p in EXP_FILE.parent.glob("*.yaml"))}'
print('ON_KAGGLE :', ON_KAGGLE)
print('code      :', CODE)
print('experiment:', EXP_FILE)

## 3. Packages
Kaggle already has PyTorch, transformers, pandas, scikit-learn and pyarrow. The project also needs
`sentencepiece` (ViSoBERT tokenizer) and `underthesea` (only for preprocessing).

In [ ]:
if ON_KAGGLE:
    pkgs = ['sentencepiece', 'underthesea'] + (['py_vncorenlp'] if INSTALL_VNCORENLP else [])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

import torch, transformers, pandas, sklearn
print(f'python {sys.version.split()[0]} | torch {torch.__version__} (cuda {torch.version.cuda}) | '
      f'transformers {transformers.__version__} | pandas {pandas.__version__} | sklearn {sklearn.__version__}')
major, minor = (int(x) for x in torch.__version__.split('.')[:2])
if (major, minor) < (2, 3):
    print('WARNING: the code was tested with torch >= 2.3 (torch.amp.GradScaler("cuda"))')

## 4. GPU check
Stops early if there is no GPU or if this PyTorch build cannot run on it.

In [ ]:
if not torch.cuda.is_available():
    print('WARNING: no GPU. Set Accelerator = GPU T4 (training on CPU is far too slow).')
else:
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    supported = f'sm_{cap[0]}{cap[1]}' in torch.cuda.get_arch_list()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {name}, compute {cap[0]}.{cap[1]}, {total / 2**30:.1f} GB, supported by this torch build: {supported}')
    if not supported:
        raise RuntimeError(f'{name} (sm_{cap[0]}{cap[1]}) is not supported by torch {torch.__version__}; switch to a T4.')
    x = torch.randn(1024, 1024, device='cuda', dtype=torch.float16)
    print('fp16 matmul OK:', bool(torch.isfinite((x @ x).sum())))

## 5. Data, selection and previous results
- `data/processed/` from the code (or regenerated with `RUN_PREPROCESS`).
- `reports/tables/selection.json` (best backbone / loss per dataset, needed from E2 on) is copied from the code if present.
- `RESUME_FROM`: copies a previous session's finished runs back, so they are skipped.

In [ ]:
if RUN_PREPROCESS:
    cmd = [sys.executable, '-m', 'utils.preprocess', '--data-root', str(CODE / 'datasets')]
    if ON_KAGGLE:
        cmd += ['--out-dir', str(WORK / 'data' / 'processed'), '--report-dir', str(WORK / 'reports' / 'tables')]
    subprocess.run(cmd, check=True)
    os.environ['DPL_PROCESSED_DIR'] = str(WORK / 'data' / 'processed' if ON_KAGGLE else CODE / 'data' / 'processed')
elif (CODE / 'data' / 'processed').is_dir():
    os.environ.setdefault('DPL_PROCESSED_DIR', str(CODE / 'data' / 'processed'))

from utils.common import models_dir, processed_dir, reports_dir
print('processed data:', processed_dir(), sorted(p.name for p in processed_dir().glob('*')))
print('models        :', models_dir())
print('reports       :', reports_dir())

selection = reports_dir() / 'tables' / 'selection.json'
if not selection.exists() and (CODE / 'reports' / 'tables' / 'selection.json').exists():
    selection.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(CODE / 'reports' / 'tables' / 'selection.json', selection)
print('selection.json:', json.loads(selection.read_text()) if selection.exists() else 'not present (needed from E2 on)')

if RESUME_FROM:
    src, n = Path(RESUME_FROM), 0
    for run in src.iterdir() if src.is_dir() else []:
        if (run / 'metrics.json').is_file() and not (models_dir() / run.name / 'metrics.json').exists():
            shutil.copytree(run, models_dir() / run.name, dirs_exist_ok=True)
            n += 1
    print(f'restored {n} finished runs from {src}')

## 6. Matrix status and run list
`trainer.analysis status`: progress of all experiments, failed runs, remaining GPU-hours and a session plan
(which `ONLY` groups fit in this session). Then `--dry-run` lists this experiment's runs.

In [ ]:
ARGS = ['--experiment', str(EXP_FILE)]
if ONLY:
    ARGS += ['--only', ONLY]
if MAX_STEPS:
    ARGS += ['--max-steps', str(MAX_STEPS)]
TRAIN = [sys.executable, '-m', 'trainer.train', *ARGS]


def run(cmd):
    """ Run a command and stream its output into the notebook. """
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            encoding='utf-8', errors='replace', env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    for line in proc.stdout:
        if 'Loading weights' not in line:
            print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {" ".join(cmd)}')


run([sys.executable, '-m', 'trainer.analysis', 'status'])
run(TRAIN + ['--dry-run'])

## 7. Train
Long-running. Each run logs one line per epoch; a failed run writes `error.txt` and the next one starts.

In [ ]:
run(TRAIN)

## 8. Results of this experiment

In [ ]:
import pandas as pd

root = models_dir() / '_debug' if MAX_STEPS else models_dir()
rows, failed = [], []
for run_path in sorted(root.glob('*__*')):
    m_file = run_path / 'metrics.json'
    if m_file.is_file():
        m = json.loads(m_file.read_text(encoding='utf-8'))
        if m.get('experiment') != EXPERIMENT:
            continue
        row = {'run_id': m['run_id'], 'best_epoch': m['best_epoch'], 'epochs': m['epochs_run'],
               'val_mF1': m['validation']['macro_f1_mean'], 'test_mF1': m['test']['macro_f1_mean']}
        for task in ('sentiment', 'topic'):
            if task in m['test']:
                row[f'test_{task}_mF1'] = m['test'][task]['macro_f1']
                row[f'test_{task}_acc'] = m['test'][task]['accuracy']
        row['minutes'] = m['train_time_s'] / 60
        row['peak_gpu_gb'] = (m['peak_gpu_mb'] or 0) / 1024
        rows.append(row)
    elif (run_path / 'error.txt').is_file():
        failed.append(run_path.name)
results = pd.DataFrame(rows)
pd.set_option('display.width', 250)
display(results.round(4) if len(results) else 'no finished runs yet')
print(f'{len(results)} finished, {len(failed)} failed: {failed}')

## 9. Update the result tables
`trainer.analysis tables` rebuilds every table from all finished runs in `models/` (partial results are fine:
rows without runs are skipped). Figures and significance tests: `python -m trainer.analysis all` (slower).

In [ ]:
run([sys.executable, '-m', 'trainer.analysis', 'tables'])

## 10. Pack the results
Zips every run folder of this experiment **without checkpoints** (`*.pt`), plus `reports/tables`
(result tables, run status, `selection.json`), for download.
Kept checkpoints (E1 `mtl/sum`, the final model) stay in `/kaggle/working/models` for E7.

In [ ]:
if ZIP_RESULTS and ON_KAGGLE and rows:
    out = WORK / f'results_{EXPERIMENT}{"_debug" if MAX_STEPS else ""}.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for r in results['run_id']:
            for f in (root / r).rglob('*'):
                if f.is_file() and f.suffix != '.pt':
                    z.write(f, f.relative_to(root.parent))
        for f in (reports_dir() / 'tables').glob('*'):
            if f.is_file():
                z.write(f, Path('reports') / 'tables' / f.name)
    print(f'wrote {out} ({out.stat().st_size / 2**20:.1f} MB)')
    kept = list(models_dir().rglob('best.pt'))
    print(f'kept checkpoints: {len(kept)} ({sum(p.stat().st_size for p in kept) / 2**30:.2f} GB)')
else:
    print('zip skipped (local run, ZIP_RESULTS = False, or no finished runs)')